# Notebook 02 - Data Preparation

This notebook covers:
- Data cleaning (missing values, outliers)
- Loops & control statements
- Pandas DataFrames
- LLM support via Claude API (bonus point)


In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd, numpy as np, re, json, requests
from pathlib import Path
pd.set_option('display.float_format', '{:.1f}'.format)
print('Imports OK')

## 1. Load Raw Data

In [ ]:
df = pd.read_csv('../data/inserate_roh.csv')
print(f'Raw data: {df.shape[0]} x {df.shape[1]}')
print('Missing values:'); print(df.isnull().sum())
df.head()

## 2. Data Cleaning with Loops & Control Statements

In [ ]:
print(f'Start: {len(df)} rows')
# Step 1: Drop missing core values
df = df.dropna(subset=['preis_chf','flaeche_m2','zimmer_anzahl','stadt'])
print(f'After dropna: {len(df)} rows')

# Step 2: Plausibility filter - FOR loop + IF/ELIF/ELSE
excluded, kept = [], []

for idx, row in df.iterrows():            # FOR loop
    reason = None

    if not (300 <= row['preis_chf'] <= 15000):        # IF
        reason = f"Price out of range: CHF {row['preis_chf']:.0f}"
    elif not (15 <= row['flaeche_m2'] <= 400):        # ELIF
        reason = f"Area out of range: {row['flaeche_m2']:.0f} m2"
    elif not (0.5 <= row['zimmer_anzahl'] <= 12):     # ELIF
        reason = f"Rooms out of range: {row['zimmer_anzahl']}"

    if reason:
        excluded.append({'index': idx, 'reason': reason})
    else:
        kept.append(idx)

df = df.loc[kept].copy()
print(f'After plausibility filter: {len(df)} rows')
print(f'Excluded: {len(excluded)} rows')
if excluded:
    print('Examples:')
    for e in excluded[:3]: print(f'  -> {e["reason"]}')

In [ ]:
# Derive new features
df['preis_pro_m2'] = (df['preis_chf'] / df['flaeche_m2']).round(2)

def price_category(p):
    if p < 1200: return 'affordable'
    elif p < 2200: return 'mid-range'
    else: return 'expensive'

def room_group(z):
    if z <= 1.5: return '1-1.5 rooms'
    elif z <= 2.5: return '2-2.5 rooms'
    elif z <= 3.5: return '3-3.5 rooms'
    else: return '4+ rooms'

df['price_category'] = df['preis_chf'].apply(price_category)
df['room_group'] = df['zimmer_anzahl'].apply(room_group)

print('New columns added:')
print(df[['preis_chf','flaeche_m2','preis_pro_m2','price_category','room_group']].head(6))

## 3. LLM Support: Classify Descriptions with Claude API

We use the Anthropic Claude API to automatically classify apartment descriptions
by amenity features. This is **Bonus Point B3**.

Claude reads the free-text description and extracts structured features:
- **balcony**: does it have a balcony or terrace?
- **parking**: does it have a parking space or garage?
- **renovated**: recently renovated or new?
- **luxury**: luxury amenities mentioned?

In [ ]:
def llm_extract_features(descriptions: list) -> list:
    """
    Uses the Anthropic Claude API to extract amenity features
    from apartment descriptions as structured JSON.
    
    Returns: list of dicts with keys: balcony, parking, renovated, luxury (bool)
    """
    batch_text = '\n---\n'.join(
        [f'[{i}] {d}' for i, d in enumerate(descriptions)]
    )
    
    prompt = f"""Analyze these {len(descriptions)} apartment descriptions.
Reply ONLY with a JSON array. Each element must have these boolean fields:
- balcony: does it have a balcony or terrace?
- parking: does it have a parking space or garage?
- renovated: recently renovated or new?
- luxury: luxury amenities mentioned?

Example: [{{\"balcony\": true, \"parking\": false, \"renovated\": true, \"luxury\": false}}, ...]

Descriptions:
{batch_text}

JSON array (exactly {len(descriptions)} elements):"""

    try:
        response = requests.post(
            'https://api.anthropic.com/v1/messages',
            headers={'Content-Type': 'application/json'},
            json={
                'model': 'claude-sonnet-4-20250514',
                'max_tokens': 1000,
                'messages': [{'role': 'user', 'content': prompt}]
            },
            timeout=30
        )
        response.raise_for_status()
        text = response.json()['content'][0]['text'].strip()
        
        # Robust JSON extraction
        json_match = re.search(r'\\[.*\\]', text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except Exception as e:
        print(f'  LLM call failed: {e}')
    
    # Fallback: all False
    return [{'balcony': False, 'parking': False, 'renovated': False, 'luxury': False}] * len(descriptions)


# Apply to listings with descriptions (in batches of 10)
print('Running LLM analysis of descriptions...')

df_with_desc = df[df['beschreibung'].notna() & (df['beschreibung'] != '')].copy()

if len(df_with_desc) > 0:
    batch_size = 10
    all_results = []
    
    for start in range(0, min(len(df_with_desc), 30), batch_size):
        batch = df_with_desc['beschreibung'].iloc[start:start+batch_size].tolist()
        results = llm_extract_features(batch)
        all_results.extend(results)
        print(f'  Batch {start//batch_size + 1}: {len(batch)} descriptions analysed')
    
    # Integrate results into DataFrame
    features_df = pd.DataFrame(all_results, index=df_with_desc.index[:len(all_results)])
    df = df.join(features_df, how='left')
    for col in ['balcony', 'parking', 'renovated', 'luxury']:
        df[col] = df[col].fillna(False)
    
    print(f'\nLLM features added:')
    print(df[['balcony', 'parking', 'renovated', 'luxury']].sum().rename('Count True'))
else:
    print('No descriptions available -> LLM step skipped')
    for col in ['balcony', 'parking', 'renovated', 'luxury']:
        df[col] = False

## 4. Save Cleaned Dataset

In [ ]:
df.to_csv('../data/inserate_bereinigt.csv', index=False)
print(f'Saved: data/inserate_bereinigt.csv')
print(f'{len(df)} rows x {len(df.columns)} columns')
print(df['price_category'].value_counts())